# RLS implementations and fixed-point arithmetic checks

保留 real/complex RLS vs RLSAlt、quantized RLS system identification/equalization、quantized LMS。浮點 δ=P₀；量化 δ=1/P₀，因此量化模型傳入 reciprocal δ 以公平比較。

來源：`Diniz-Chap5-Impltest.ipynb`。所有資料與結果留在記憶體，圖表使用共用深色風格。

執行順序：設定 → 參數與資料 → 模型與訓練 → 圖表與數值摘要。

## 1. Setup

In [ ]:
from pathlib import Path
import sys
from functools import partial

import numpy as np

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "adsp").is_dir())
if str(project_root) not in sys.path:
  sys.path.insert(0, str(project_root))

from adsp import data, training, statistics as stats
from adsp import plotting as plot
from adsp import filters as af
from adsp import equalization as ae
from adsp.datasets import SampleBatch

plot.use_dark()

## 2. Equivalent RLS recursions on complex data

In [ ]:
h = np.array([0.8 + 0.2j, -0.4 + 0.1j, 0.25 - 0.1j, 0.1])
make_data = partial(data.system_identification, 2000, h=h, input_kind="complex_gaussian", noise_var=1e-4)
models = {cls.__name__: partial(cls, filter_order=3, lam=0.99, delta=0.5) for cls in (af.RLS, af.RLSAlt)}
results = training.compare(models, make_data, n_trials=1, seed=50)
assert np.allclose(results["RLS"].representative.y, results["RLSAlt"].representative.y, atol=1e-9)
plot.learning(results, window=31)
plot.show()
training.summary(results)

## 3. Floating and fixed-point system identification

In [ ]:
make_real = partial(data.system_identification, 600, h=np.array([0.8, -0.4, 0.25, 0.1]), noise_var=1e-4)
models = {
  "RLS float": partial(af.RLSAlt, filter_order=3, lam=0.99, delta=0.5),
  "RLS fixed": partial(af.QuantizedRLSAlt, filter_order=3, lam=0.99, delta=2.0, q_total_bits=16, q_frac_bits=12),
}
fixed_results = training.compare(models, make_real, n_trials=1, seed=51)
plot.learning(fixed_results, window=31)
plot.show()
training.summary(fixed_results)

## 4. Equalization: float/fixed RLS and fixed LMS

In [ ]:
h = np.array([0.9, 0.3, -0.2])
make_equalization = partial(data.equalization, 2000, h=h, decision_delay=2, symbol_kind="bpsk", noise_var=1e-4)
models = {
  "RLS float": partial(af.RLSAlt, filter_order=5, lam=0.995, delta=0.5),
  "RLS fixed": partial(af.QuantizedRLSAlt, filter_order=5, lam=0.995, delta=2.0, q_total_bits=16, q_frac_bits=12),
  "LMS fixed": partial(af.QuantizedLMS, filter_order=5, mu=0.05, q_total_bits=16, q_frac_bits=12),
}
eq_results = training.compare(models, make_equalization, n_trials=1, seed=52)
plot.learning(eq_results, window=50)
plot.show()
plot.stem({
  name: np.convolve(h, np.conj(res.representative.w_final)).real
  for name, res in eq_results.items()
}, title="Cascade impulse response, shared quantization formats")
plot.show()
training.summary(eq_results, tail=500)